[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C39_Distributed_Training_Course/05_orchestration_debug/05_orchestration_debug.ipynb)

# 05 · 编排与调试（用 numpy 在单进程内模拟多 rank）

**本课没有真实集群**：我们在单个 python 进程里用 numpy/stdlib **模拟** N 个 rank，把编排与调试的核心结构——rank↔坐标映射、通信组、落后者、扩展效率、重叠、死锁检测——都跑出来并用 `assert` 验证。

路线：rank/坐标映射 → 落后者木桶效应 → 强扩展(Amdahl) → 弱扩展 → 通信-计算重叠 → 死锁检测 → ✏️ 练习 → 📖 答案 → 🧪 MFU 真实胶囊 → 🔧 torchrun/SLURM 旁注。

> 心智模型：**一个 rank = 列表里的一个下标；一个通信组 = 一组下标；同步 = 对这组下标的某种聚合（max / sum）**。

## 1 · rank ↔ (dp, tp, pp) 坐标与通信组

多维并行下，每个 rank 在一个 `(dp, tp, pp)` 网格里有坐标。约定展平公式（最内 TP、其次 DP、最外 PP）：

$$\text{rank} = pp\cdot(n_{dp} n_{tp}) + dp\cdot n_{tp} + tp$$

通信规则：**沿哪个维度并行，就在固定其余两维、遍历该维的那组 rank 上做 collective**。

In [ ]:
import numpy as np
from itertools import product
rng = np.random.default_rng(0)

def rank_of(dp, tp, pp, n_dp, n_tp):
    return pp * (n_dp * n_tp) + dp * n_tp + tp

def coords_of(rank, n_dp, n_tp):
    tp = rank % n_tp
    dp = (rank // n_tp) % n_dp
    pp = rank // (n_dp * n_tp)
    return dp, tp, pp

n_dp, n_tp, n_pp = 2, 2, 4
world = n_dp * n_tp * n_pp
print(f'world_size = {n_dp}*{n_tp}*{n_pp} = {world}')
# 往返一致：rank -> 坐标 -> rank
for r in range(world):
    dp, tp, pp = coords_of(r, n_dp, n_tp)
    assert rank_of(dp, tp, pp, n_dp, n_tp) == r
# 坐标 -> rank -> 坐标
for dp, tp, pp in product(range(n_dp), range(n_tp), range(n_pp)):
    assert coords_of(rank_of(dp, tp, pp, n_dp, n_tp), n_dp, n_tp) == (dp, tp, pp)
print('✅ rank ↔ (dp,tp,pp) 往返一致（16 个 rank 全部对上）')

现在按规则**枚举出每种并行的通信组**：DP 组（固定 tp,pp 变 dp）、TP 组（固定 dp,pp 变 tp）、PP 组（固定 dp,tp 变 pp）。

In [ ]:
def dp_groups(n_dp, n_tp, n_pp):
    '''每个 DP 组 = 固定 (tp,pp)、遍历 dp 的 rank 列表。'''
    groups = []
    for pp in range(n_pp):
        for tp in range(n_tp):
            groups.append([rank_of(dp, tp, pp, n_dp, n_tp) for dp in range(n_dp)])
    return groups

def tp_groups(n_dp, n_tp, n_pp):
    groups = []
    for pp in range(n_pp):
        for dp in range(n_dp):
            groups.append([rank_of(dp, tp, pp, n_dp, n_tp) for tp in range(n_tp)])
    return groups

def pp_groups(n_dp, n_tp, n_pp):
    groups = []
    for dp in range(n_dp):
        for tp in range(n_tp):
            groups.append([rank_of(dp, tp, pp, n_dp, n_tp) for pp in range(n_pp)])
    return groups

dpg = dp_groups(n_dp, n_tp, n_pp)
tpg = tp_groups(n_dp, n_tp, n_pp)
ppg = pp_groups(n_dp, n_tp, n_pp)
print('DP 组（每组 2 个 rank，应有 8 组）:', dpg)
print('TP 组（每组 2 个 rank，应有 8 组）:', tpg)
print('PP 组（每组 4 个 rank，应有 4 组）:', ppg)
# 校验：每个 rank 恰好属于每种组各一次；组大小正确
for groups, size, ngrp in [(dpg, n_dp, world//n_dp), (tpg, n_tp, world//n_tp), (ppg, n_pp, world//n_pp)]:
    assert len(groups) == ngrp
    assert all(len(g) == size for g in groups)
    flat = sorted(r for g in groups for r in g)
    assert flat == list(range(world)), '每个 rank 应恰属于该类组中的一个'
print('✅ DP/TP/PP 通信组划分正确：每个 rank 在每类组里恰好出现一次')

## 2 · 落后者：同步训练的木桶效应

同步数据并行每步以 all-reduce 收尾，这是一道**屏障**：必须等所有 rank 到齐。
所以单步耗时 = **最慢 rank** 的耗时：$t_{\text{step}} = \max_r t_r \ge \bar t$。

我们模拟每个 rank 的单步耗时，量化一个落后者造成的**浪费的 GPU-秒** = $\sum_r (\max - t_r)$。

In [ ]:
def sync_step_time(per_rank_times):
    '''同步屏障：整步耗时 = 最慢 rank。返回 (step_time, 浪费的GPU秒, 利用率)。'''
    t = np.asarray(per_rank_times, dtype=float)
    step = t.max()                      # 屏障 = 等最慢的
    wasted = float((step - t).sum())    # 其余 rank 在屏障前空等的总时长
    util = float(t.mean() / step)       # 平均忙 / 实际墙钟
    return float(step), wasted, util

N = 16
# 健康情形：所有 rank ~1.0（轻微抖动）
healthy = 1.0 + 0.02 * rng.standard_normal(N)
# 落后者情形：15 个 ~1.0，但 7 号卡慢 30%
straggler = healthy.copy(); straggler[7] = 1.30

for name, ts in [('健康', healthy), ('有落后者', straggler)]:
    step, wasted, util = sync_step_time(ts)
    print(f'{name:7s}: step={step:.3f}s  浪费={wasted:5.2f} GPU·s  利用率={util:.1%}')

step_h, _, util_h = sync_step_time(healthy)
step_s, wasted_s, util_s = sync_step_time(straggler)
assert step_s > step_h, '一个落后者抬高整步耗时'
assert util_s < util_h, '落后者拉低利用率'
assert wasted_s > 3.0, '15 个 rank 各等 ~0.3s，浪费约 4.5 GPU·s'
print('\n✅ 单个 rank 慢 30% → 全体每步慢 ~30%，其余 15 卡白等。这就是木桶效应。')

## 3 · 强扩展与 Amdahl 定律

问题固定、加卡求快。可并行比例 `p`、串行比例 `1−p`（分布式里主要是通信税），则：

$$S(N) = \frac{1}{(1-p) + p/N}, \qquad E(N) = S(N)/N$$

残酷的上界：$N\to\infty$ 时 $S \to 1/(1-p)$。下面把曲线打出来。

In [ ]:
def amdahl_speedup(N, p):
    return 1.0 / ((1.0 - p) + p / N)

def amdahl_efficiency(N, p):
    return amdahl_speedup(N, p) / N

Ns = [1, 8, 64, 512]
print(f"{'N':>5} | {'S@p=.95':>9} {'E@.95':>7} | {'S@p=.99':>9} {'E@.99':>7}")
for N in Ns:
    s95, e95 = amdahl_speedup(N, .95), amdahl_efficiency(N, .95)
    s99, e99 = amdahl_speedup(N, .99), amdahl_efficiency(N, .99)
    print(f'{N:>5} | {s95:>8.2f}x {e95:>6.1%} | {s99:>8.2f}x {e99:>6.1%}')

# 效率随 N 单调下降
effs = [amdahl_efficiency(N, .95) for N in [1, 2, 4, 8, 16, 32, 64]]
assert all(effs[i] > effs[i+1] for i in range(len(effs)-1)), '强扩展效率随 N 递减'
# 加速比有天花板 1/(1-p)
assert amdahl_speedup(10**9, .95) < 20.0001
assert abs(amdahl_speedup(10**9, .95) - 20.0) < 1e-3, 'p=.95 上限 = 1/.05 = 20x'
# p 越高，同样 N 下加速比越高
assert amdahl_speedup(512, .99) > amdahl_speedup(512, .95)
print('\n✅ 5% 串行就把加速比锁死在 20x；把 p 从 .95 提到 .99，512 卡加速比 19x→84x。')

## 4 · 弱扩展：每卡工作量固定

大模型预训练是**弱扩展**：加卡时把总 batch 等比放大，**每卡**工作量不变，理想是每步耗时恒定。

但通信开销随 N 增长（如 ring all-reduce 的延迟项随组大小、或跨节点 ~$\log N$）。弱扩展效率 = `理想恒定时间 / 实际时间`。

In [ ]:
def weak_scaling_time(N, compute_per_step=1.0, comm_unit=0.02, growth='log'):
    '''每卡计算量固定=compute_per_step；通信随 N 增长。返回该规模下单步耗时。'''
    if growth == 'log':
        comm = comm_unit * np.log2(N)      # 跨节点树形/分层 ~ log N
    elif growth == 'const':
        comm = comm_unit                   # 理想 ring all-reduce：与 N 几乎无关
    else:
        comm = comm_unit * N
    return compute_per_step + comm

def weak_efficiency(N, **kw):
    ideal = weak_scaling_time(1, **kw)    # N=1 基准（注意 log2(1)=0）
    return ideal / weak_scaling_time(N, **kw)

print(f"{'N':>5} | {'step(log)':>10} {'eff(log)':>9} | {'eff(const)':>11}")
for N in [1, 8, 64, 512]:
    t_log = weak_scaling_time(N, growth='log')
    e_log = weak_efficiency(N, growth='log')
    e_const = weak_efficiency(N, growth='const')
    print(f'{N:>5} | {t_log:>10.3f} {e_log:>8.1%} | {e_const:>10.1%}')

# log 增长：效率随 N 缓慢下降，但远好于强扩展
assert weak_efficiency(512, growth='log') > 0.84, '弱扩展+log通信仍应保持高效率'
assert weak_efficiency(8, growth='log') > weak_efficiency(512, growth='log')
# 通信若与 N 无关（理想 ring），弱扩展效率恒为 1
assert abs(weak_efficiency(512, growth='const') - 1.0) < 1e-9
print('\n✅ 弱扩展是万卡训练能成立的原因：每卡工作量不变，效率随 N 只缓降。')

## 5 · 通信-计算重叠

朴素：`step = compute + comm`（先算完再通信）。
重叠：梯度反向时一层层就绪，可边算边异步通信 → `step = max(compute, comm)`（只要通信被算力盖住就近乎免费）。

In [ ]:
def step_serial(compute, comm):
    return compute + comm

def step_overlapped(compute, comm):
    return max(compute, comm)            # 理想重叠：取大者

compute, comm = 1.0, 0.4
ser = step_serial(compute, comm)
ovl = step_overlapped(compute, comm)
speedup = ser / ovl
print(f'compute={compute}, comm={comm}')
print(f'串行 step = {ser:.2f}   重叠 step = {ovl:.2f}   加速 {speedup:.2f}x')

# 效率定义：有用计算 / 墙钟
eff_ser = compute / ser
eff_ovl = compute / ovl
print(f'利用率：串行 {eff_ser:.1%}  →  重叠 {eff_ovl:.1%}')
assert ovl < ser, '重叠应更快'
assert eff_ovl > eff_ser, '重叠提升利用率'
# 当通信完全被计算盖住(comm<=compute)，重叠后通信免费、利用率=100%
assert abs(eff_ovl - 1.0) < 1e-9, 'comm<=compute 时重叠把通信藏到 100% 利用率'
# 但若通信比计算还久，重叠也只能到 max，通信成新瓶颈
assert step_overlapped(1.0, 2.0) == 2.0
print('\n✅ 通信能被计算盖住时近乎免费；盖不住时通信成为新下界。这是 DDP 梯度分桶的核心。')

## 6 · 死锁检测：collective 必须对齐

集合通信的铁律：**所有 rank 以一致顺序调用一致的 collective**。
若某 rank 在某步调了和别人不同的 collective（或漏调/多调），就会在屏障前永远等下去（直到 NCCL timeout）。

给每个 rank 一串 collective「标签」序列，写一个检测器找出**第一处对不齐**的步号。

In [ ]:
def find_collective_mismatch(rank_traces):
    '''rank_traces[r] = 第 r 个 rank 依次调用的 collective 标签列表。
       返回第一处所有 rank 不一致的步号；全程一致返回 -1。'''
    n_ranks = len(rank_traces)
    max_len = max(len(t) for t in rank_traces)
    for step in range(max_len):
        # 收集每个 rank 在这一步调的 collective（越界=已退出/漏调，记为 None）
        calls = [t[step] if step < len(t) else None for t in rank_traces]
        if len(set(calls)) != 1:        # 不是所有 rank 都调了同一个 → 卡死
            return step, calls
    return -1, None

# 对齐的情形：3 个 rank 每步都一致
good = [['all_reduce', 'all_gather', 'all_reduce']] * 3
step, calls = find_collective_mismatch(good)
print(f'对齐情形: mismatch step = {step}')
assert step == -1, '完全对齐不应有死锁'

# 植入 bug：rank 2 在第 1 步多走了一个数据相关分支(broadcast)，导致和别人错位
bad = [['all_reduce', 'all_gather', 'all_reduce'],
       ['all_reduce', 'all_gather', 'all_reduce'],
       ['all_reduce', 'broadcast', 'all_gather', 'all_reduce']]
step, calls = find_collective_mismatch(bad)
print(f'植入bug情形: 第 {step} 步对不齐 → {calls}')
assert step == 1, 'rank2 在第 1 步调了 broadcast，其余调 all_gather'
assert calls[2] == 'broadcast' and calls[0] == 'all_gather'
print('✅ 检测器定位到 rank 2 在第 1 步与其余 rank 对不齐——这正是 flight recorder 做的事。')

---
## ✏️ 练习 1：反推坐标 + 找出某 rank 的所有组员

并行配置 `n_dp=4, n_tp=2, n_pp=2`（world=16）。实现 `groupmates(rank, n_dp, n_tp, n_pp)`：返回字典 `{'dp': [...], 'tp': [...], 'pp': [...]}`，分别是与该 rank **同 DP / 同 TP / 同 PP 组**的 rank 列表（含自己）。

提示：先用 `coords_of` 求出本 rank 的 (dp,tp,pp)，再分别固定两维、遍历第三维。

In [ ]:
def groupmates(rank, n_dp, n_tp, n_pp):
    # TODO: 求本 rank 坐标 (dp,tp,pp)，再构造三个组：
    #   dp 组：固定本 (tp,pp)，dp 取遍 0..n_dp-1
    #   tp 组：固定本 (dp,pp)，tp 取遍 0..n_tp-1
    #   pp 组：固定本 (dp,tp)，pp 取遍 0..n_pp-1
    #   用 rank_of(...) 把坐标变回 rank。返回 {'dp':..,'tp':..,'pp':..}
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
n_dp, n_tp, n_pp = 4, 2, 2
g = groupmates(5, n_dp, n_tp, n_pp)
assert set(g.keys()) == {'dp', 'tp', 'pp'}
assert len(g['dp']) == n_dp and len(g['tp']) == n_tp and len(g['pp']) == n_pp
assert 5 in g['dp'] and 5 in g['tp'] and 5 in g['pp'], '组里应含自己'
# rank 5 = coords (dp=2,tp=1,pp=0): tp 组应是同 (dp=2,pp=0) 的 tp=0,1 → rank 4,5
assert sorted(g['tp']) == [4, 5]
# 每个组员反查坐标应与自己只在对应维不同
dp0, tp0, pp0 = coords_of(5, n_dp, n_tp)
for r in g['dp']:
    d, t, pp = coords_of(r, n_dp, n_tp)
    assert (t, pp) == (tp0, pp0), 'dp 组内只有 dp 维变化'
print('✅ 练习 1 通过：能正确算出任意 rank 的 DP/TP/PP 组员')

## ✏️ 练习 2：落后者诊断 —— 浪费占比与 p99/均值

给每个 rank 的单步耗时数组，实现 `straggler_report(times)` 返回字典：
- `wasted_frac`：浪费的 GPU·秒 占「总 GPU·秒（= N×step）」的比例；
- `p99_over_mean`：单步耗时的 99 分位 ÷ 均值（>1 越多说明尾巴越重）。

用 `np.percentile(times, 99)` 取 p99。

In [ ]:
def straggler_report(times):
    # TODO: step=max(times); wasted=sum(step-times);
    #   total_gpu_s = len(times)*step; wasted_frac = wasted/total_gpu_s
    #   p99 = np.percentile(times,99); p99_over_mean = p99/mean(times)
    #   返回 {'wasted_frac':..,'p99_over_mean':..}
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
t_even = np.ones(100)                       # 完全均衡
rep_even = straggler_report(t_even)
assert abs(rep_even['wasted_frac']) < 1e-9, '均衡时零浪费'
assert abs(rep_even['p99_over_mean'] - 1.0) < 1e-9

t_bad = np.ones(100); t_bad[-5:] = 3.0      # 5 个严重落后者(尾巴重)
rep_bad = straggler_report(t_bad)
# step=3, wasted=95*(3-1)=190, total=100*3=300 → 0.6333
assert abs(rep_bad['wasted_frac'] - 190/300) < 1e-9
assert rep_bad['p99_over_mean'] > 2.0, '尾巴重时 p99 远超均值'
print(f"均衡: 浪费占比={rep_even['wasted_frac']:.1%}  p99/mean={rep_even['p99_over_mean']:.2f}")
print(f"落后: 浪费占比={rep_bad['wasted_frac']:.1%}  p99/mean={rep_bad['p99_over_mean']:.2f}")
print('✅ 练习 2 通过：能用浪费占比 + p99/均值量化落后者严重程度')

## ✏️ 练习 3：反解 Amdahl 的串行比例

你在 `N` 卡上**实测**到效率 `E`（= 加速比/N）。反推这次训练的串行（通信税）比例 `p_serial = 1−p`。

从 $E = \dfrac{1}{N(1-p)+p}$ 解出 $p$，再令 `p_serial = 1-p`。实现 `infer_serial_fraction(N, E)`。

（推导：$E\,[N(1-p)+p]=1 \Rightarrow p = \dfrac{1/E - N}{1 - N}$。）

In [ ]:
def infer_serial_fraction(N, E):
    # TODO: 由 E = 1/(N(1-p)+p) 解 p，返回串行比例 1-p
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
# 正演一遍再反解，应能还原
for p_true in [0.90, 0.95, 0.99]:
    for N in [8, 64, 512]:
        E = amdahl_efficiency(N, p_true)
        serial = infer_serial_fraction(N, E)
        assert abs(serial - (1 - p_true)) < 1e-9, f'应还原串行比例 {1-p_true}'
# 一个「实测」例子：64 卡只拿到 30% 效率，通信税多大？
serial = infer_serial_fraction(64, 0.30)
print(f'64 卡、实测效率 30% → 串行(通信)比例 ≈ {serial:.1%}')
assert 0 < serial < 1
print('✅ 练习 3 通过：能从实测效率反推通信税，指导该不该继续加卡')

## ✏️ 练习 4：瓶颈分类器

给一次训练的三个测量值：`compute`（计算段耗时）、`comm`（通信段，已扣除可重叠部分）、`straggler_gap`（最慢 rank 比中位 rank 多出的时间）。

实现 `classify_bottleneck(compute, comm, straggler_gap)` 返回 `'compute'` / `'comm'` / `'straggler'`：
**取三者中最大的那一项**作为主瓶颈（简化判据）。

In [ ]:
def classify_bottleneck(compute, comm, straggler_gap):
    # TODO: 在 {'compute':compute,'comm':comm,'straggler':straggler_gap} 里取最大值对应的键
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
assert classify_bottleneck(1.0, 0.2, 0.1) == 'compute'    # 算力受限(好状态)
assert classify_bottleneck(1.0, 1.5, 0.1) == 'comm'       # 通信受限
assert classify_bottleneck(1.0, 0.2, 1.3) == 'straggler'  # 落后者受限
# 处方应随分类不同
RX = {'compute': '换更强卡/提升kernel效率', 'comm': '重叠/压缩/改并行切法', 'straggler': '均衡数据/剔除坏节点'}
label = classify_bottleneck(1.0, 1.5, 0.1)
print(f'诊断 = {label} → 处方：{RX[label]}')
print('✅ 练习 4 通过：把「慢在哪」固化成可执行的诊断→处方')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def groupmates(rank, n_dp, n_tp, n_pp):
    dp, tp, pp = coords_of(rank, n_dp, n_tp)
    return {
        'dp': [rank_of(d, tp, pp, n_dp, n_tp) for d in range(n_dp)],
        'tp': [rank_of(dp, t, pp, n_dp, n_tp) for t in range(n_tp)],
        'pp': [rank_of(dp, tp, q, n_dp, n_tp) for q in range(n_pp)],
    }

In [ ]:
# 练习 2 参考答案
def straggler_report(times):
    t = np.asarray(times, dtype=float)
    step = t.max()
    wasted = float((step - t).sum())
    total_gpu_s = len(t) * step
    return {
        'wasted_frac': wasted / total_gpu_s,
        'p99_over_mean': float(np.percentile(t, 99) / t.mean()),
    }

In [ ]:
# 练习 3 参考答案
def infer_serial_fraction(N, E):
    # E = 1/(N(1-p)+p) → 1/E = N(1-p)+p = N - p(N-1) → p = (N - 1/E)/(N-1)
    p = (N - 1.0 / E) / (N - 1.0)
    return 1.0 - p

In [ ]:
# 练习 4 参考答案
def classify_bottleneck(compute, comm, straggler_gap):
    d = {'compute': compute, 'comm': comm, 'straggler': straggler_gap}
    return max(d, key=d.get)

---
## 🧪 真实数据胶囊：算一次真实训练的 MFU

**MFU（Model FLOPs Utilization）** = 实际有用算力 ÷ 硬件峰值。稠密 Transformer 每 token 约 `6N` FLOPs（前向 2N + 反向 4N）：

$$\text{MFU} = \frac{6N \cdot (\text{tokens/s})}{\text{GPU 数} \times \text{单卡峰值 FLOP/s}}$$

下面用接近真实公开报告的数字（GPT-3 175B 量级 / A100、H100 峰值）算 MFU，并验证它落在合理区间。

In [ ]:
def mfu(num_params, tokens_per_sec, num_gpus, peak_flops_per_gpu):
    '''返回 model FLOPs utilization (0..1)。'''
    achieved = 6.0 * num_params * tokens_per_sec     # 有用 FLOP/s
    peak = num_gpus * peak_flops_per_gpu             # 理论峰值 FLOP/s
    return achieved / peak

# 真实量级的硬件峰值（公开规格，bf16/fp16 张量核心，约数）
PEAK = {
    'A100': 312e12,    # 312 TFLOP/s bf16
    'H100': 989e12,    # 989 TFLOP/s bf16 (稠密)
}

try:
    # 一个接近真实的大规模训练设定（175B 模型，1024×A100）
    N_params = 175e9
    n_gpus = 1024
    # 实测吞吐：175B 这种大模型每卡 token 吞吐很低（每 token 要 6N≈1.05e12 FLOPs）。
    # 每卡 ~140 tok/s 对应 ~47% MFU，与 GPT-3 级公开报告同量级。
    toks_per_s = 1024 * 140           # ≈ 每卡 ~140 tok/s，总 ~143k tok/s
    m = mfu(N_params, toks_per_s, n_gpus, PEAK['A100'])
    print(f'175B / {n_gpus}×A100 / {toks_per_s:,} tok/s → MFU = {m:.1%}')
except Exception as e:
    print('离线回退：使用硬编码真实数值', e)
    m = 0.471
    print(f'175B / 1024×A100（硬编码）→ MFU ≈ {m:.1%}')

assert 0.2 < m < 0.7, '大规模训练 MFU 通常在 20%~60%'
print('对照：精调的大规模预训练 MFU 多在 ~35%~60%；<15% 说明有明显浪费可挖。')

**🧪 胶囊练习**：同一个 175B 训练，换到 1024×**H100**（峰值更高）。若 MFU 想保持不变，需要的吞吐 tokens/s 是多少？实现 `tokens_for_target_mfu(num_params, target_mfu, num_gpus, peak_flops)`（反解上面的 MFU 公式）。

In [ ]:
def tokens_for_target_mfu(num_params, target_mfu, num_gpus, peak_flops):
    # TODO: 由 MFU = 6N*toks / (gpus*peak) 解 toks
    raise NotImplementedError

In [ ]:
# 自测
tgt = 0.471
toks_h100 = tokens_for_target_mfu(175e9, tgt, 1024, PEAK['H100'])
toks_a100 = tokens_for_target_mfu(175e9, tgt, 1024, PEAK['A100'])
# H100 峰值 ≈ A100 的 3.17 倍 → 同 MFU 需 ~3.17 倍吞吐
ratio = PEAK['H100'] / PEAK['A100']
assert abs(mfu(175e9, toks_h100, 1024, PEAK['H100']) - tgt) < 1e-6, '反解应自洽'
assert abs(toks_h100 / toks_a100 - ratio) < 1e-6, '更强卡要按峰值比例提高吞吐才能维持同 MFU'
print(f'1024×H100 维持 MFU={tgt:.1%} 需 ≈ {toks_h100:,.0f} tok/s（约 A100 设定的 {ratio:.2f}×）')
print('✅ 胶囊练习通过：MFU 是相对峰值的，换更强的卡，要更高吞吐才能维持同一利用率')

In [ ]:
# 📖 胶囊参考答案
def tokens_for_target_mfu(num_params, target_mfu, num_gpus, peak_flops):
    return target_mfu * num_gpus * peak_flops / (6.0 * num_params)

---
## 🔧 旁注：真实集群怎么把它拉起来（torchrun / SLURM，本环境不跑）

本模块在单进程里模拟了 rank/组/落后者/扩展效率。在真实集群上，把脚本拉起到多卡多机靠**启动器**——下面是对照（**伪代码，本环境不跑**）：

**单机 8 卡（torchrun）：**
```bash
torchrun --standalone --nproc_per_node=8 train.py
# torchrun 为每个进程注入 RANK / LOCAL_RANK / WORLD_SIZE / MASTER_ADDR / MASTER_PORT
```

**2 机 × 8 卡（torchrun + rendezvous）：**
```bash
# 每台机器各跑一次，--node_rank 不同；rdzv_endpoint 指向 rank0 机器
torchrun --nnodes=2 --nproc_per_node=8 --node_rank=0 \
         --rdzv_id=job42 --rdzv_backend=c10d --rdzv_endpoint=10.0.0.1:29500 train.py
```

**SLURM 作业脚本（生产 HPC）：**
```bash
#SBATCH --nodes=2
#SBATCH --ntasks-per-node=8       # 每节点 8 个进程 = 8 卡
#SBATCH --gpus-per-node=8
srun python train.py             # SLURM 注入 SLURM_PROCID 等，再映射成 rank/local_rank
```

**训练脚本里对应的几行（伪代码）：**
```python
import os, torch, torch.distributed as dist
rank       = int(os.environ['RANK'])
local_rank = int(os.environ['LOCAL_RANK'])
world_size = int(os.environ['WORLD_SIZE'])
torch.cuda.set_device(local_rank)          # ← 用 local_rank 绑卡，不是 rank！
dist.init_process_group('nccl')            # rendezvous + 建全局组
# 按并行维度建子组（对应 worked 1 的 dp/tp/pp groups）：
# dp_group = dist.new_group(ranks=[...])    # 梯度 all-reduce 在此组
# tp_group = dist.new_group(ranks=[...])    # 激活 all-reduce 在此组
```

对应关系：`os.environ['RANK']` ↔ worked 1 的线性 rank；`new_group(ranks=[...])` ↔ 我们枚举出的 dp/tp/pp 组；`set_device(local_rank)` ↔ 进程到 GPU 的绑定。**调试就是当这套东西卡住时，用 flight recorder（worked 6 的检测器）+ 每 rank 日志重建谁在等谁。**

### 小结
- **编排** = 把 (rank, world_size, local_rank, MASTER_ADDR/PORT) 正确分发给每个进程；`local_rank` 绑卡、`rank` 参与全局 collective。
- **通信组**：沿哪个维度并行就在那个维度的 rank 组里做 collective；建组错 = **沉默地**算错。
- **落后者**：同步 = 木桶效应，$t_{\text{step}}=\max_r t_r$；一个慢 rank 拖垮全体，N 越大越常见。
- **扩展效率**：强扩展受 Amdahl 锁死（$1/(1-p)$ 上界）；弱扩展靠每卡工作量不变撑起万卡；**重叠**把通信藏进计算。
- **MFU** 一把尺子量穿所有低效；**调试**靠每 rank 日志 + flight recorder + py-spy 重建全局现场。

🎓 **恭喜——你已走完 C39 全部六个模块**：从集合通信原语，到数据/张量/流水并行，到容错，再到这里的编排与调试。你现在能用 numpy 把分布式训练的每个核心机制从零推一遍、对拍验证，并把验证过的逻辑映射到真实的 `torch.distributed` / FSDP / Megatron 代码。